In [1]:
import torch
import numpy as np
import os
from tokenizers import Tokenizer
from tqdm import tqdm
import json
import math
from collections import defaultdict
from itertools import islice
from pathlib import Path

In [2]:
torch.manual_seed(0)

In [3]:
TOKENIZER = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/tokenizer.json"

PRETRAIN_TRAIN = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/pretrain_train.bin"
PRETRAIN_VALIDATION = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/pretrain_valid.bin"

SFT_TRAIN_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_tokens.bin"
SFT_TRAIN_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_mask.bin"
SFT_TRAIN_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_offsets.npy"

SFT_VALIDATION_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_tokens.bin"
SFT_VALIDATION_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_mask.bin"
SFT_VALIDATION_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_offsets.npy"

SFT_TEST_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_tokens.bin"
SFT_TEST_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_mask.bin"
SFT_TEST_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_offsets.npy"

In [4]:
import shutil, time

t0 = time.time()
shutil.copy(PRETRAIN_TRAIN, "/tmp/pretrain_train.bin")
shutil.copy(PRETRAIN_VALIDATION, "/tmp/pretrain_valid.bin")
PRETRAIN_TRAIN = "/tmp/pretrain_train.bin"
PRETRAIN_VALIDATION = "/tmp/pretrain_valid.bin"
print(f"скопировано за {time.time() - t0:.0f} с")

скопировано за 59 с


In [5]:
class PretrainDataset(torch.utils.data.Dataset):

    def __init__(self, path, context_window_tokens):
        self.path = path
        self.context_window_tokens = context_window_tokens
        self.length = os.path.getsize(path) // 2 // self.context_window_tokens
        self.data = None

    def __getitem__(self, index):
        if self.data is None:
            self.data = np.memmap(self.path, dtype=np.uint16, mode="r")
        sample_substring = self.data[self.context_window_tokens * index:self.context_window_tokens  * (index+1)]
        return torch.from_numpy(sample_substring.astype(np.int64))
    def __len__(self):
        return self.length

In [6]:
DATA_DIR = "/kaggle/input/datasets/absolute1000/tokenized-llm-training"

max_ctx, max_ans = 0, 0
for split in ["train", "valid", "test"]:
    mask    = np.fromfile(f"{DATA_DIR}/sft_{split}_mask.bin", dtype=np.uint8)
    offsets = np.load(f"{DATA_DIR}/sft_{split}_offsets.npy")

    d = np.diff(mask.astype(np.int8), prepend=0, append=0)
    starts, ends = np.where(d == 1)[0], np.where(d == -1)[0]

    ans = ends - starts
    ctx = starts - offsets[np.searchsorted(offsets, starts, side="right") - 1]

    print(f"{split}: ответов {len(ans)}, макс ответ {ans.max()}, макс контекст {ctx.max()}")
    max_ctx, max_ans = max(max_ctx, ctx.max()), max(max_ans, ans.max())

print(f"\nMAX_CTX = {max_ctx}\nMAX_ANS = {max_ans}")

train: ответов 153100, макс ответ 380, макс контекст 929
valid: ответов 2377, макс ответ 167, макс контекст 622
test: ответов 2471, макс ответ 140, макс контекст 468

MAX_CTX = 929
MAX_ANS = 380


In [7]:
def collate_fn(list_of_samples):
    #tok.encode("<pad>").ids = [0]
    
    B = len(list_of_samples)
    _batch_samples = torch.zeros((B, 1000), dtype=torch.int64)
    _batch_targets = torch.zeros((B, 500), dtype=torch.int64)
    for idx, data in enumerate(list_of_samples):
        context, bot_tokens = data
        _batch_samples[idx, -len(context):] = context
        _batch_targets[idx, :len(bot_tokens)] = bot_tokens
    return _batch_samples, _batch_targets

In [8]:
def add_noise(tokens, t):
    #tok.encode("<mask>").ids = 1
    #t[samples]
    batch_len, tokens_per_sample = tokens.shape
    noise = torch.rand(batch_len, tokens_per_sample, device=tokens.device) #noise[samples, answer_window]
    noise = noise < t.unsqueeze(-1)
    masked = torch.where(noise, 1, tokens)
    return masked, noise

In [9]:
def Loss(predictions, ans, noise, answer_window, t):
    #predictions[samples, 12288, answer_window], ans[samples, answer_window], noise[samples, answer_window]
    #answer_window[1], t[samples]
    loss = torch.nn.functional.cross_entropy(predictions, ans, reduction='none') #loss[samples, answer_window]
    loss = torch.where(noise, loss, 0) #loss[samples, answer_window]
    loss = torch.sum(loss, dim=1) #loss[samples]
    loss /= t
    loss /= answer_window
    return torch.mean(loss)

In [10]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

In [11]:
class NN(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [12]:
def make_record(
    reason, step, cur_epoch, model, optimizer, scheduler, scaler, best_loss,
    context_size, answer_size, train_loss_history, valid_loss_history, MODEL_SETTINGS_CONFIG
):
    path = f'/kaggle/working/PRETRAIN_{reason}.pt'
    Path(path).unlink(missing_ok=True)
    torch.save({
        'step': step,
        'epoch':cur_epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_loss': best_loss,
        'context_size': context_size,
        'answer_size': answer_size,
        'train_loss_history': dict(train_loss_history),
        'valid_loss_history': dict(valid_loss_history),
        'MODEL_SETTINGS_CONFIG': MODEL_SETTINGS_CONFIG
    }, path)

In [13]:
MODEL_SETTINGS_CONFIG = {
    "embedding_size": 384,
    "encoder_num": 6,
    "heads_encoder": 6,
    "thinking_internal_state_size": 128,
    "decoder_num": 2,
    "heads_decoder": 6,
}

In [14]:
def TRAIN_PIPELINE_PRETRAIN_MODE(
    MAX_STEPS=None,
    START_STEP=0,
    BATCH_SIZE=64,
    context_size = None,
    answer_size = None,
    lr = 3e-4,
    weight_decay = 0.1,
    warmup_steps = 500,
    best_loss = float('inf'),
    config_path = None,
    BATCH_PER_EPOCH=1064
):
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpt = torch.load(config_path, map_location=DEVICE, weights_only=False) if config_path else None
    config = ckpt['MODEL_SETTINGS_CONFIG'] if ckpt else MODEL_SETTINGS_CONFIG
    model = NN(**config).to(DEVICE)
    decay    = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2] 
    optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": weight_decay},
     {"params": no_decay, "weight_decay": 0.0}], lr=lr)
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_STEPS - warmup_steps)
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])
    if ckpt:
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        START_STEP = ckpt['step'] + 1
        best_loss = ckpt['best_loss']
        context_size = ckpt['context_size']
        answer_size = ckpt['answer_size']
    
    def get_data_from_batch(batch):
        context1, context2 = batch[:, :context_size].to(DEVICE), batch[:, context_size:].to(DEVICE)
        target2 = context2.clone()
        noise_schedule = ((torch.arange(BATCH_SIZE, device=DEVICE) + torch.rand(1, device=DEVICE)) / BATCH_SIZE).clamp(min=1e-3)
        context2, noise2 = add_noise(context2, noise_schedule)
        return context1, context2, noise2, noise_schedule, target2
    def infinite(loader):
        while True:
            for batch in loader:
                yield batch

    TRAIN_DATALOADER = torch.utils.data.DataLoader(
        PretrainDataset(PRETRAIN_TRAIN, context_size + answer_size), BATCH_SIZE, shuffle=True, num_workers=4,
        pin_memory=True, drop_last=True, persistent_workers=True
    ) 
    VALIDATION_DATALOADER = torch.utils.data.DataLoader(
        PretrainDataset(PRETRAIN_VALIDATION, context_size + answer_size), BATCH_SIZE, num_workers=4,
        pin_memory=True, drop_last=True, persistent_workers=True
    )

    train_loss_history = defaultdict(list, ckpt['train_loss_history']) if ckpt else defaultdict(list)
    valid_loss_history = defaultdict(list, ckpt['valid_loss_history']) if ckpt else defaultdict(list)
    TRAIN_CYCLE = tqdm(range(START_STEP, MAX_STEPS), desc="TRAIN", mininterval=10)
    train_iter = infinite(TRAIN_DATALOADER)
    for step in TRAIN_CYCLE:
        cur_epoch = step // BATCH_PER_EPOCH
        model.train()
        batch = next(train_iter)
        context1, context2, noise2, noise_schedule, target2 = get_data_from_batch(batch)
                   
        with torch.amp.autocast('cuda'):
            out = model(context1, context2).transpose(1, 2)
            loss = Loss(out, target2, noise2, answer_size, noise_schedule)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        if step % 50 == 0:
            TRAIN_CYCLE.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
        train_loss_history[cur_epoch].append(loss.item())
        
        #VALIDATION CYCLE
        if step % BATCH_PER_EPOCH == 0 and step != 0:
            VALIDATION_CYCLE = tqdm(
                islice(VALIDATION_DATALOADER, 100), total=100, 
                desc=f"VALIDATION, Epoch {cur_epoch}", leave=False, mininterval=5
            )
            model.eval()
            with torch.no_grad():
                for batch in VALIDATION_CYCLE:
                    context1, context2, noise2, noise_schedule, target2 = get_data_from_batch(batch)
                
                    with torch.amp.autocast('cuda'):
                        out = model(context1, context2).transpose(1, 2)
                        loss = Loss(out, target2, noise2, answer_size, noise_schedule)
                    valid_loss_history[cur_epoch].append(loss.item())
                else:
                    make_record(
                        "RECORD", step, cur_epoch, model, optimizer, scheduler, scaler,
                        best_loss, context_size, answer_size, train_loss_history, valid_loss_history, config
                    )

            mean_val_loss_per_epoch = torch.tensor(valid_loss_history[cur_epoch]).mean()
            print(f"step {step}: val_loss = {mean_val_loss_per_epoch:.4f}")
            if mean_val_loss_per_epoch < best_loss:
                best_loss = mean_val_loss_per_epoch.item()
                make_record(
                    "BEST", step, cur_epoch, model, optimizer, scheduler, scaler, best_loss, 
                    context_size, answer_size, train_loss_history, valid_loss_history, config
                    )
    make_record("RECORD", step, cur_epoch, model, optimizer, scheduler, scaler, best_loss, context_size, answer_size, train_loss_history, valid_loss_history, config)
    return model

In [15]:
model = TRAIN_PIPELINE_PRETRAIN_MODE(
    MAX_STEPS=50000,
    START_STEP=0,
    BATCH_SIZE=96,
    context_size = 256,
    answer_size = 128,
    lr = 3e-4,
    weight_decay = 0.1,
    warmup_steps = 500,
    best_loss = float('inf'),
    config_path = None,
    BATCH_PER_EPOCH=1064
)

VALIDATION, Epoch 1:  95%|█████████▌| 95/100 [00:20<00:01,  4.61it/s]
                                                                     

step 1064: val_loss = 6.4679


VALIDATION, Epoch 2:  96%|█████████▌| 96/100 [00:20<00:00,  4.61it/s]
                                                                     

step 2128: val_loss = 5.9727


VALIDATION, Epoch 3:  96%|█████████▌| 96/100 [00:20<00:00,  4.61it/s]
                                                                     

...

TRAIN:  96%|█████████▌| 47881/50000 [8:17:56<36:55,  1.05s/it, loss=4.3879, lr=1.39e-06]

step 47880: val_loss = 4.3830


TRAIN:  98%|█████████▊| 48945/50000 [8:29:00<18:25,  1.05s/it, loss=4.3884, lr=3.65e-07]

step 48944: val_loss = 4.3805


TRAIN: 100%|██████████| 50000/50000 [8:39:36<00:00,  1.60it/s, loss=4.4210, lr=7.25e-10]


In [16]:
torch.save(model.state_dict(), f'/kaggle/working/FINAL_MODEL.pt')

In [17]:
import math, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
CKPT = "/kaggle/working/PRETRAIN_BEST.pt"

ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
best_model = NN(**ckpt["MODEL_SETTINGS_CONFIG"]).to(DEVICE)
print(best_model.load_state_dict(ckpt["model"]))
best_model.eval()

has_thinking = any(k.startswith("Thinking.") for k in ckpt["model"])
n_params = sum(p.numel() for p in best_model.parameters()) / 1e6
print(f"{'С Thinking' if has_thinking else 'БЕЗ Thinking'} | шаг {ckpt['step']} | best val {ckpt['best_loss']:.4f} | {n_params:.2f}M")

@torch.no_grad()
def generate(model, text, answer_len=64, temperature=0.7, top_k=50, block=8, rep_penalty=1.3, context_len=256):
    ids = tok.encode(text).ids[-context_len:]
    ctx = torch.tensor([ids], device=DEVICE)
    memory, mask = model.encode(ctx)
    ans = torch.full((1, answer_len), MASK_ID, dtype=torch.long, device=DEVICE)
    pos = torch.arange(answer_len, device=DEVICE)

    for step in range(answer_len * 2):
        is_mask = ans[0] == MASK_ID
        n_left = int(is_mask.sum())
        if n_left == 0:
            break
        logits = model.decode(ans, memory, mask)[0].float()
        logits[:, MASK_ID] = -float("inf")
        if rep_penalty > 1.0:
            used = ans[0][~is_mask].unique()
            if len(used):
                lu = logits[:, used]
                logits[:, used] = torch.where(lu > 0, lu / rep_penalty, lu * rep_penalty)
        logits = logits / temperature
        kth = logits.topk(top_k, dim=-1).values[:, -1:]
        probs = torch.softmax(logits.masked_fill(logits < kth, -float("inf")), -1)
        pred = torch.multinomial(probs, 1).squeeze(-1)
        conf = probs.gather(-1, pred[:, None]).squeeze(-1)

        first = int(is_mask.nonzero()[0])
        eligible = is_mask & (pos < (first // block + 1) * block)
        conf = conf.masked_fill(~eligible, -1.0)
        k = min(max(1, math.ceil(n_left / max(1, answer_len - step))), int(eligible.sum()))
        idx = conf.topk(k).indices
        ans[0, idx] = pred[idx]

    out = ans[0].tolist()
    if EOS_ID in out:
        out = out[:out.index(EOS_ID)]
    return tok.decode(out)

prompts = [
    "The history of the Roman Empire began",
    "Once upon a time, in a small village,",
    "To make a good cup of coffee, you need",
    "Scientists have recently discovered that",
    "The main reason why people move to big cities is",
    "Hi! How are you doing today?",
]

for p in prompts:
    torch.manual_seed(SEED)
    print("=" * 100)
    print("ПРОМПТ:", p)
    print(generate(best_model, p))

<All keys matched successfully>
БЕЗ Thinking | шаг 45752 | best val 4.3796 | 20.06M
ПРОМПТ: The history of the Roman Empire began


NameError: name 'tok' is not defined